# Week 3 Assignment: Python & Data Wrangling

The supplied course archive contained only the brief, so this is a synthetic practice sales dataset. The workflow demonstrates cleaning, filtering, derived columns, summaries, and charts.

## 1. Load and inspect

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
raw = pd.read_csv(Path('data') / 'messy_sales.csv')
print('Shape:', raw.shape)
display(raw.head())
display(raw.isna().sum().to_frame('missing_values'))

## 2. Clean and transform

In [ ]:
df=raw.copy(); df.columns=df.columns.str.strip().str.lower()
for c in ['customer','region','product','status','email']: df[c]=df[c].astype('string').str.strip()
for c in ['region','product','status']: df[c]=df[c].str.title()
df['customer']=df.customer.replace('',pd.NA); df['email']=df.email.replace('',pd.NA)
df['order_date']=pd.to_datetime(df.order_date,errors='coerce')
df['quantity']=pd.to_numeric(df.quantity,errors='coerce'); df['unit_price']=pd.to_numeric(df.unit_price,errors='coerce')
df['order_id']=df.order_id.astype('string').str.strip()
df=df.dropna(subset=['order_id','order_date','quantity','unit_price'])
df=df[(df.quantity>0)&(df.unit_price>0)].drop_duplicates('order_id',keep='first')
df['customer']=df.customer.fillna('Unknown'); df['email']=df.email.fillna('Not provided')
df['revenue']=df.quantity*df.unit_price; df['order_month']=df.order_date.dt.to_period('M').astype(str); df['is_paid']=df.status.eq('Paid')
df=df.sort_values(['order_date','order_id']).reset_index(drop=True)
print('Rows after cleaning:',len(df)); display(df.head())

## 3. Filter and summarize

In [ ]:
paid=df[df.is_paid].copy()
summary=df.groupby('status').agg(order_count=('order_id','count'),total_revenue=('revenue','sum')).reset_index()
display(summary)
print(f'Total cleaned revenue: {df.revenue.sum():,.2f}')
print(f'Paid revenue: {paid.revenue.sum():,.2f}')

## 4. Visualize paid revenue

In [ ]:
sns.set_theme(style='whitegrid')
by_region=paid.groupby('region',as_index=False).revenue.sum()
sns.barplot(data=by_region,x='region',y='revenue',color='steelblue'); plt.title('Paid Revenue by Region'); plt.tight_layout(); plt.show()
by_month=paid.groupby('order_month',as_index=False).revenue.sum()
sns.lineplot(data=by_month,x='order_month',y='revenue',marker='o',color='steelblue'); plt.title('Paid Revenue by Month'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 5. Export results

In [ ]:
out=Path('outputs'); out.mkdir(exist_ok=True)
df.to_csv(out/'cleaned_sales.csv',index=False); paid.to_csv(out/'paid_orders.csv',index=False); summary.to_csv(out/'summary_by_status.csv',index=False)
print('CSV outputs saved to outputs/')

## Conclusion

This assignment demonstrates CSV loading, quality inspection, text normalization, type conversion, missing/invalid value handling, filtering, derived columns, grouped summaries, and introductory visualization. Use the actual course dataset instead if your instructor provided one separately.